<a href="https://colab.research.google.com/github/AlaaTaima/plant-model/blob/main/plantmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# وفي أي جلسة لاحقة: لا تشغّل الجدول والتنظيف والتقسيم، بل الربط والتحميل ثم خلية قراءة الملفات الثلاثة (11) ثم من الإعدادات و الترميز وما بعدها.
from google.colab import drive
drive.mount('/content/drive')

import os
color_dir = "/content/drive/MyDrive/plant_project/plantvillage dataset/color"
print(os.path.exists(color_dir), len(os.listdir(color_dir)))   # True 38

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True 38


In [2]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_a19db74358b723fba726eb9370551b93"

!pip install -q "kaggle>=1.8.0"
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content
!unzip -q /content/plantvillage-dataset.zip "plantvillage dataset/color/*" -d /content

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:15<00:00, 146MB/s] 



In [3]:
import os
LOCAL = "/content/plantvillage dataset/color"
classes = sorted(os.listdir(LOCAL))
total = sum(len(os.listdir(os.path.join(LOCAL, c))) for c in classes)
print(len(classes), total)

38 54305


In [4]:
#الدوال و المسار

#تعرّف الدالة info(item) التي تفحص صورة واحدة: تقرأ الملف مرة واحدة وتحسب منه بصمة
#MD5، ثم تتحقق من سلامته بـ verify()، وتستخرج الأبعاد ونمط الألوان. إذا فشل شيء تضع corrupted=True

import os, io, hashlib, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

LOCAL = "/content/plantvillage dataset/color"
classes = sorted(os.listdir(LOCAL))
print("عدد الفئات:", len(classes))

def info(item):
    p, c = item
    row = {"path": p, "label": c, "fname": os.path.basename(p),
           "bytes": os.path.getsize(p), "width": None, "height": None,
           "mode": None, "corrupted": False, "hash": None}
    try:
        with open(p, "rb") as f:
            data = f.read()                      # قراءة واحدة فقط
        row["hash"] = hashlib.md5(data).hexdigest()
        Image.open(io.BytesIO(data)).verify()
        im = Image.open(io.BytesIO(data))
        row["width"], row["height"] = im.size
        row["mode"] = im.mode
    except Exception:
        row["corrupted"] = True
    return row

عدد الفئات: 38


In [5]:

#بناء df الجدول وحفظه
#تجهز قائمة بكل الصور، وتشغّل info على كل منها بـ 8 threads بالتوازي، وتجمع النتائج في جدول df (54305 صفاً × 9 أعمدة)، وتحفظه في
#df_raw.csv على Drive.
items = [(os.path.join(LOCAL, c, f), c)
         for c in classes for f in os.listdir(os.path.join(LOCAL, c))]

with ThreadPoolExecutor(8) as ex:
    rows = list(tqdm(ex.map(info, items), total=len(items)))

df = pd.DataFrame(rows)
df.to_csv("/content/drive/MyDrive/plant_project/df_raw.csv", index=False)
print(df.shape)   # المتوقع (54305, 9)

  0%|          | 0/54305 [00:00<?, ?it/s]

(54305, 9)


In [6]:
#التنظيف والحفظ
#تحدد الصور المستبعدة (bad): التالفة،
#والنسخ المكررة بالبايت (تبقي أول نسخة)، والصور التي تظهر تحت أكثر من فئة. ثم تبني clean_df (54284 صورة) وتحفظه في clean_df.csv.
df = df.sort_values("path").reset_index(drop=True)
ok = df[~df["corrupted"]]
dup = ok.duplicated(subset="hash", keep="first")
n_labels = ok.groupby("hash")["label"].nunique()
conflict = n_labels[n_labels > 1].index

bad = set(df.loc[df["corrupted"], "path"])
bad |= set(ok.loc[dup, "path"])
bad |= set(ok.loc[ok["hash"].isin(conflict), "path"])

clean_df = df[~df["path"].isin(bad)].drop(columns="corrupted").reset_index(drop=True)
print(len(df), "|", len(bad), "|", len(clean_df))

clean_df.to_csv("/content/drive/MyDrive/plant_project/clean_df.csv", index=False)

54305 | 21 | 54284


In [7]:
#التقسيم: تدريب تحقق تيست

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    clean_df, test_size=0.30, stratify=clean_df["label"], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

train_df, val_df, test_df = (x.reset_index(drop=True) for x in (train_df, val_df, test_df))

print("الأحجام:", len(train_df), len(val_df), len(test_df))
print("الفئات في كل قسم:", train_df["label"].nunique(), val_df["label"].nunique(), test_df["label"].nunique())
print("أقل عدد صور لفئة في test:", test_df["label"].value_counts().min())

p = "/content/drive/MyDrive/plant_project/"
train_df.to_csv(p + "train.csv", index=False)
val_df.to_csv(p + "val.csv", index=False)
test_df.to_csv(p + "test.csv", index=False)

الأحجام: 37998 8143 8143
الفئات في كل قسم: 38 38 38
أقل عدد صور لفئة في test: 23


In [18]:
#فحص الـ GPU،
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [19]:
#تقرأ ملفات CSV الثلاثة من Drive. هي خلية للاستخدام بعد انقطاع الجلسة، فلا تعيد الفحص والتقسيم.
#خلية 11 بعد التجميع و التقسيم : الملفات الثلاثة بتكون ع درايف ف بنشغلها هي فقط عند انقطاع الجلسة
import pandas as pd
d = "/content/drive/MyDrive/plant_project/"
train_df = pd.read_csv(d + "train.csv")
val_df = pd.read_csv(d + "val.csv")
test_df = pd.read_csv(d + "test.csv")
print(len(train_df), len(val_df), len(test_df))


37998 8143 8143


In [9]:
#الإعدادات وترميز الفئات

#IMG = 128: كل صورة تُصغَّر إلى 128×128.
#BATCH = 64: عدد الصور في الدفعة الواحدة.
#class2idx يحول اسم كل فئة إلى رقم من 0 إلى 37، وتُضاف الأرقام في عمود y، لأن الشبكة تتعامل مع أرقام لا نصوص.

import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG = 128
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

classes = sorted(train_df["label"].unique())
class2idx = {c: i for i, c in enumerate(classes)}
for d in (train_df, val_df, test_df):
    d["y"] = d["label"].map(class2idx)
NUM_CLASSES = len(classes)
print(NUM_CLASSES)   # 38

38


In [10]:
#class_weight
from sklearn.utils.class_weight import compute_class_weight

w = compute_class_weight("balanced",
                         classes=np.arange(NUM_CLASSES),
                         y=train_df["y"].values)
class_weight = dict(enumerate(w))
print(round(w.min(), 2), round(w.max(), 2))

0.26 9.43


In [11]:
#pipeline الصور (tf.data)
#load تقرأ الملف وتفك ترميز الـ JPEG إلى 3 قنوات وتصغّرها إلى 128×128، والتطبيع يتم لاحقاً داخل النموذج.
#make_ds تبني dataset من المسارات والتسميات، مع خلط (shuffle) للتدريب فقط، ثم map لتحميل الصور بالتوازي، ثم تقسيم لدفعات، وprefetch لتجهيز الدفعة التالية أثناء تدريب الحالية.
#الخلط يتم قبل تحميل الصور (على المسارات فقط)، فيكون سريعاً وخفيفاً على الذاكرة.
#val وtest بدون خلط، فيبقى ترتيبهما مطابقاً لترتيب test_df، وهذا ضروري للمقارنة مع y_true في التقييم.
def load(path, y):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))
    return img, y          # التطبيع يتم داخل المودل

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["y"].values))
    if training:
        ds = ds.shuffle(len(df), seed=42)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds   = make_ds(val_df, False)
test_ds  = make_ds(test_df, False)

In [12]:
#بناء المودل CNN

#طبقات augmentation داخل النموذج (قلب، تدوير، تكبير، تباين)، وتعمل أثناء التدريب فقط وتتوقف في التقييم تلقائياً.
#Rescaling(1/255) للتطبيع إلى [0, 1].
#4 كتل بعدد فلاتر 32 ثم 64 ثم 128 ثم 256، وكل كتلة: Conv → BatchNorm → Conv → BatchNorm → MaxPool → Dropout(0.25). كل كتلة تصغّر الحجم المكاني للنصف وتزيد عمق الميزات.
#GlobalAveragePooling2D بدل Flatten: تأخذ متوسط كل feature map، فتقلل المعاملات كثيراً.
#Dense(256) + Dropout(0.5) ثم Dense(38, softmax) للإخراج.
#compile: المحسّن Adam بمعدل تعلم 0.001، والخسارة sparse_categorical_crossentropy لأن التسميات أرقام صحيحة وليست one-hot.
#summary() تعرض الطبقات وعدد المعاملات.


augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augment")

inputs = keras.Input((IMG, IMG, 3))
x = augment(inputs)
x = layers.Rescaling(1./255)(x)

for filters in (32, 64, 128, 256):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

cnn = keras.Model(inputs, outputs)
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augment (Sequential)            │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 256)    │       295,16

 Total params: 1,251,654 (4.77 MB)

 Trainable params: 1,249,734 (4.77 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [20]:
#التدريب ()
#ModelCheckpoint: يحفظ أفضل نموذج (أعلى val_accuracy) في Drive.
#EarlyStopping: يوقف التدريب إذا لم تتحسن val_accuracy خلال 5 epochs، ويعيد أوزان أفضل epoch.
#ReduceLROnPlateau: يخفّض معدل التعلم للنصف إذا لم تتحسن val_loss خلال epochين.
#ثم fit لمدة حتى 30 epoch.
cb2 = [
    keras.callbacks.ModelCheckpoint("/content/drive/MyDrive/plant_project/cnn_best_cw.keras",
                                    monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
]
h_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=30,
                callbacks=cb2, class_weight=class_weight)

Epoch 1/30
 71/594 ━━━━━━━━━━━━━━━━━━━━ 1:43 198ms/step - accuracy: 0.9708 - loss: 0.0786

KeyboardInterrupt: 

In [ ]:
#validation
from sklearn.metrics import f1_score, classification_report

pred = np.argmax(cnn.predict(val_ds), axis=1)
y_val = val_df["y"].values
print("val accuracy :", (pred == y_val).mean())
print("val macro F1 :", f1_score(y_val, pred, average="macro"))
print(classification_report(y_val, pred, target_names=classes, digits=3))

In [ ]:
#التقييم النهائي على test (مرة واحدة فقط)
#تحمّل أفضل نموذج محفوظ، وتقيس الدقة على test
#ثم تتنبأ بكل صوره وتطبع
# classification_report: الدقة (precision) والاسترجاع
 #(recall) وF1 لكل فئة. كتبت أن التقييم "مرة واحدة فقط" وهذا صحيح، لأن تكرار الضبط بناءً على نتيجة test يحوّلها إلى validation ويفقدها قيمتها.

from sklearn.metrics import classification_report

cnn = keras.models.load_model("/content/drive/MyDrive/plant_project/cnn_best_cw.keras")

loss, acc = cnn.evaluate(test_ds)
print("Test accuracy:", acc)

y_true = test_df["y"].values
y_pred = np.argmax(cnn.predict(test_ds), axis=1)
print(classification_report(y_true, y_pred, target_names=classes, digits=3))

121/121 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - accuracy: 0.9461 - loss: 0.1639
Test accuracy: 0.9461101293563843
121/121 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab      0.817     0.978     0.890        91
                                 Apple___Black_rot      0.897     1.000     0.946        87
                          Apple___Cedar_apple_rust      0.976     0.976     0.976        42
                                   Apple___healthy      0.801     0.987     0.885       233
                               Blueberry___healthy      0.989     0.841     0.909       214
          Cherry_(including_sour)___Powdery_mildew      0.986     0.932     0.958       148
                 Cherry_(including_sour)___healthy      0.968     1.000     0.984       122
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot      0.943     0.892     0.917        74
                   

In [ ]:
# شغّل هذا الكود في Colab (بعد انتهاء التدريب) لتصدير ملفي المودل اللذين يحتاجهما السيرفر.
# يعمل في خلية Colab عادية: انسخ المحتوى كله في خلية واحدة.

import json, shutil
import tensorflow as tf
from tensorflow import keras
from google.colab import files

print("TensorFlow version:", tf.__version__)   # ضع هذا الرقم في requirements.txt

# 1) أفضل نسخة من المودل
cnn = keras.models.load_model("/content/drive/MyDrive/plant_project/cnn_best.keras")
print("input shape:", cnn.input_shape)         # يجب أن يكون (None, 128, 128, 3)
shutil.copy("/content/drive/MyDrive/plant_project/cnn_best.keras", "/content/cnn_best.keras")

# 2) أسماء الفئات بنفس ترتيب التدريب (مهم جداً)
classes = sorted(train_df["label"].unique())
assert len(classes) == 38
with open("/content/classes.json", "w", encoding="utf-8") as f:
    json.dump(classes, f, ensure_ascii=False, indent=2)

# 3) تنزيل الملفين إلى جهازك
files.download("/content/cnn_best.keras")
files.download("/content/classes.json")


TensorFlow version: 2.21.0
input shape: (None, 128, 128, 3)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>